# Alert Stream and Visits 

Query the number of diaSources reported in `lsst.prompt.prod.numDiaSourcesGood` and merge with visit information.

In [ ]:
import os
from astropy.time import Time, TimeDelta
from IPython.display import display, HTML
from rubin_nights import connections
from rubin_nights.influx_query import InfluxQueryClient
import rubin_nights.dayobs_utils as rn_dayobs

import numpy as np
import pandas as pd 

import rubin_sim.maf as maf
from lsst_survey_sim import plot

import matplotlib.pyplot as plt

import rubin_nights.plot_utils as rn_plots
colors = rn_plots.PlotStyles.band_colors
markers = rn_plots.PlotStyles.band_symbols

import logging
logging.getLogger('rubin_nights').setLevel(logging.INFO)

In [ ]:
# Connect using your tokenfile and appropriate site 
tokenfile = os.path.join(os.path.expanduser("~"), ".lsst/usdf_rsp")
site = "usdf"
endpoints = connections.get_clients(tokenfile, site)

In [ ]:
day_obs = 20251114
# Find start and end of the night
tstart, tend = rn_dayobs.day_obs_sunset_sunrise(day_obs, sun_alt=-12)

# OR .. 
tstart = Time("2025-11-09T12:00:00")
tend = Time.now()

sasquatch = InfluxQueryClient("usdfdev", db_name="lsst.prompt")
tt = sasquatch.select_time_series("lsst.prompt.prod.numDiaSourcesGood", 
                                  ["visit", "band", "day_obs", "detector", "dataset_tag", "numAllDiaSources", "numGoodDiaSources", "run"], 
                                  tstart, tend)
if len(tt) > 0:
    alertsum = tt.groupby("visit").agg({'numAllDiaSources': 'sum', 'numGoodDiaSources': ('sum', 'median'), 'detector': 'count'})
    alertsum.index = alertsum.index.astype(int)
    cols = [f"{c[0]}_{c[1]}" for c in alertsum.columns]
    alertsum = alertsum.droplevel(level=0, axis=1)
    alertsum.columns = cols
    alertsum.rename({"detector_count": "nDiaDetectors_count"}, axis=1, inplace=True)
else:
    print("no alerts")

In [ ]:
fields = ['visit', 'exposure', 'detector', 'day_obs', 'ra', 'dec', 'band', 'nPsfStar', 'psfArea', 'psfSigma', 'skyBg', 'skyNoise', 'zeroPoint']
#tt2 = sasquatch.select_time_series("lsst.prompt.prod.initialPviSummaryMetrics", fields, tstart, tend)

In [ ]:
programs = ["BLOCK-365", "BLOCK-407", "BLOCK-408", "BLOCK-416"]
constraint = " or ".join([f"science_program = '{p}'" for p in programs])
visits = endpoints['consdb'].get_visits("lsstcam", tstart, tend, visit_constraint=constraint)

In [ ]:
print("number of visits:", len(visits))
print("number of visits with alerts:", len(alertsum))

In [ ]:
if len(visits) > 0 and len(alertsum) > 0:
    vv = pd.merge(visits, alertsum, left_on='visit_id', right_index=True, how='outer')
    cols = ['visit_id', 'observation_reason', 'obs_start', 'band', 's_ra', 's_dec', 'sky_rotation', 'clouds', 
        'fwhm_eff', 'cat_m5', 'numGoodDiaSources_sum', 'numGoodDiaSources_median', 'nDiaDetectors_count']
    #display(vv[cols])

In [ ]:
run_calc = True
if run_calc:
    q = vv.query("numGoodDiaSources_sum > 0")
    nvisits = {}
    m_nvis = maf.CountMetric(col='obs_start_mjd', metric_name = "Nvisits")
    s = maf.HealpixSlicer(nside=64, lon_col='s_ra', lat_col='s_dec', rot_sky_pos_col_name = 'sky_rotation')
    for b in ['u', 'g', 'r', 'i', 'z', 'y', 'all']:
        constraint = f"{b}"
        if b == 'all':
            opsvis = q.to_records()
        else:
            opsvis = q.query("band == @b").to_records()
        nvisits[b] = maf.MetricBundle(m_nvis, s, constraint)
        g = maf.MetricBundleGroup({f'nvisits {b}': nvisits[b],}, None)
        if len(opsvis) > 0:
            g.run_current(constraint, opsvis)    

    background = plot.get_background(nside=64)
    
    fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(16, 10),)
    axdict = {"u": ax[0][0], "g": ax[0][1], "r": ax[0][2],
              "i": ax[1][0], "z": ax[1][1], "y": ax[1][2], "all": None}
    for b in ["u", "g", "r", "i", "z", "y"]:
        if nvisits[b].metric_values is not None:
            if len(nvisits[b].metric_values.compressed()) > 1:
                vmax = np.percentile(nvisits[b].metric_values.compressed(), 95)
            else:
                vmax = None
            label_dec = False
            if b == 'u' or b == 'i':
                label_dec = True
            fig = plot.make_plot(nvisits[b], background=background, proj='McBryde', vmax=vmax, ax=axdict[b], title=f"LSSTCam band with Alerts {b}", label_dec=label_dec)
    fig.tight_layout()
    
    vmax = np.percentile(nvisits['all'].metric_values.compressed(), 95)
    fig = plot.make_plot(nvisits['all'], background=background, proj='mcbryde', vmin=None, vmax=vmax, ax=None, title=f"LSSTCam visits with Alerts")

In [ ]:
if len(vv) > 0:
    #vv = vv.query("not observation_reason.str.contains('ddf')")
    fig, ax = plt.subplots(1, 2, figsize=(18, 8))
    for tt in ['ddf', 'pair', 'too', 'other']:
        if tt == 'other':
            qq = vv.query("~observation_reason.str.contains('pair') and ~observation_reason.str.contains('ddf') and ~observation_reason.str.contains('too')")
        elif tt == 'too': 
            qq = vv.query("observation_reason.str.contains('too')")
        else:
            qq = vv.query("observation_reason.str.contains(@tt)")
        if tt == 'ddf':
            marker='^'
        else:
            marker = 'o'
        for b in qq.band.unique():
            dq = qq.query("band == @b")
            for d in dq.day_obs.unique():
                q = dq.query("day_obs == @d and cat_m5 >0 and numGoodDiaSources_sum > 0")
                if len(q) > 0:
                    if not np.all(np.isnan(q.cat_m5)) and not np.all(np.isnan(q.numGoodDiaSources_sum)):
                        scatter = ax[0].scatter(q.cat_m5, q.numGoodDiaSources_sum, s=q.nDiaDetectors_count/189*20 + 5,
                                                marker=marker, linestyle='', label=f"{b} {tt} {d}")

    # ax[0].legend()
    ax[0].set_xlabel("estimated limiting magnitude", fontsize='large')
    ax[0].set_ylabel("nGoodDiaSources per visit", fontsize='large')
    day_title = vv.day_obs.unique()
    if len(day_title) == 1:
        day_title = day_title[0]
    else:
        day_title = f"{day_title.min()} - {day_title.max()}"
    ax[0].set_title(f"Dayobs {day_title}", fontsize='large')
    

    for tt in ['ddf', 'pair', 'other']:
        if tt == 'other':
            qq = vv.query("~observation_reason.str.contains('pair') and ~observation_reason.str.contains('ddf')")
        else:
            qq = vv.query("observation_reason.str.contains(@tt)")
        if tt == 'ddf':
            marker='^'
        else:
            marker = 'o'
        for b in qq.band.unique():
            dq = qq.query("band == @b")
            for d in dq.day_obs.unique():
                q = dq.query("day_obs == @d")
                if len(q) > 0:
                    if not np.all(np.isnan(q.cat_m5)) and not np.all(np.isnan(q.numGoodDiaSources_sum)):
                        ax[1].scatter(q.nDiaDetectors_count, q.numGoodDiaSources_sum, s=(q.cat_m5 - q.cat_m5.min())*5 + 9, #color=colors[b], 
                                      marker=marker, linestyle='', label=f"{b} {tt} {d}")
    ax[1].legend(loc=(1.01, 0.1))
    ax[1].set_xlabel("nDiaDetectors per visit")
    ax[1].set_ylabel("nGoodDiaSources per visit", fontsize='large')
    ax[1].set_title(f"Dayobs {day_title}", fontsize='large')

In [ ]:
q = vv.query("day_obs == 20251116 and numGoodDiaSources_sum>0")

fig, ax = plt.subplots(6, 1, figsize=(8, 12), sharex=True)

leg_x = 1.01
leg_y = 0.2
fig.subplots_adjust(hspace=0.1)
i = 0
for b in q.band.unique():
    qq = q.query("band == @b")
    ax[i].plot(qq.obs_start_mjd, qq.fwhm_eff, linestyle='', marker=markers[b], color=colors[b], label=b)
ax[i].plot(q.obs_start_mjd, q.fwhm_eff, '-', color='gray', alpha=0.3, zorder=0)
ax[i].legend(loc=(leg_x, leg_y))
ax[i].set_ylabel("fwhm (arcsecond)")
ax[i].grid(alpha=0.3)
i += 1
for b in q.band.unique():
    qq = q.query("band == @b")
    ax[i].plot(qq.obs_start_mjd, qq.clouds, linestyle='', marker=markers[b], color=colors[b], label=b)
ax[i].plot(q.obs_start_mjd, q.clouds, '-', color='gray', alpha=0.3, zorder=0)
ax[i].legend(loc=(leg_x, leg_y))
ax[i].set_ylabel("cloud extinction")
ax[i].grid(alpha=0.3)
i += 1
for b in q.band.unique():
    qq = q.query("band == @b")
    ax[i].plot(qq.obs_start_mjd, qq.cat_m5, linestyle='', marker=markers[b], color=colors[b], label=b)
ax[i].plot(q.obs_start_mjd, q.cat_m5, '-', color='gray', alpha=0.3, zorder=0)
ax[i].legend(loc=(leg_x, leg_y))
ax[i].set_ylabel("estimated m5")
ax[i].grid(alpha=0.3)
i += 1
for b in q.band.unique():
    qq = q.query("band == @b")
    ax[i].plot(qq.obs_start_mjd, qq.numGoodDiaSources_median*189, linestyle='', marker='o', color='none', markeredgecolor=colors[b])
    ax[i].plot(qq.obs_start_mjd, qq.numGoodDiaSources_sum, linestyle='', marker=markers[b], color=colors[b], label=b)
ax[i].plot(q.obs_start_mjd, q.numGoodDiaSources_sum, '-', color='gray', alpha=0.3, zorder=0)
ax[i].legend(loc=(leg_x, leg_y))
ax[i].set_ylabel("Number DiaSources")
ax[i].set_ylim(-10, max(vv.numGoodDiaSources_sum.max() * 1.2, vv.numGoodDiaSources_median.max() * 189 * 1.2))
ax[i].grid(alpha=0.3)
i += 1
for b in q.band.unique():
    qq = q.query("band == @b")
    ax[i].plot(qq.obs_start_mjd, qq.nDiaDetectors_count, linestyle='', marker=markers[b], color=colors[b], label=b)
ax[i].plot(q.obs_start_mjd, q.nDiaDetectors_count, '-', color='gray', alpha=0.3, zorder=0)
ax[i].legend(loc=(leg_x, leg_y))
ax[i].set_ylabel("NDetectors")
ax[i].set_ylim(0, 189)
ax[i].grid(alpha=0.3)
i += 1
for b in q.band.unique():
    qq = q.query("band == @b")
    ax[i].plot(qq.obs_start_mjd, qq.eclip_lat, linestyle='', marker=markers[b], color=colors[b], label=b)
ax[i].plot(q.obs_start_mjd, q.eclip_lat, '-', color='gray', alpha=0.3, zorder=0)
ax[i].legend(loc=(leg_x, leg_y))
ax[i].set_ylabel("Ecliptic Latitude")
ax[i].grid(alpha=0.3)
day_title = q.day_obs.unique()
if len(day_title) == 1:
    day_title = day_title[0]
else:
    day_title = f"{day_title.min()} - {day_title.max()}"
_ = fig.suptitle(f"DayObs {day_title}", y=0.92)